# VeriSight: FaceForensics++ Dual-Stream Deepfake Detection
### Spatial Visual Textures + 2D FFT Frequency Analysis on Official FaceForensics++ (FF++) Benchmark

**Why FaceForensics++?**
FaceForensics++ (TUM) is the gold standard academic benchmark for facial manipulation. It provides authentic video frames alongside 5 advanced manipulation techniques:
1. **Deepfakes:** Autoencoder-based latent facial replacement
2. **Face2Face:** Real-time facial reenactment modifying expressions and mouth motion
3. **FaceSwap:** Classical 3D model-based face transfer
4. **FaceShifter:** High-fidelity identity replacement with adaptive occlusion handling
5. **NeuralTextures:** Neural rendering using neural texture maps and photometric loss

**Architecture:**
- **Spatial Stream:** Pretrained ResNet-50 (or ResNet-18) detecting facial blending seams, boundary warping, and skin texture mismatches.
- **Spectral Stream:** On-the-fly 2D Fast Fourier Transform (2D FFT) + Convolutional Frequency Encoder to isolate periodic upsampling artifacts and frequency grid anomalies.
- **Multi-Task Fusion:** Combines both streams with auxiliary loss supervision to prevent overfitting.


In [ ]:
# Step 1: Verify Hardware & GPU Acceleration
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM Allocated:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("Warning: GPU not active. Navigate to Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
# Step 2: Install and Import Required Libraries
!pip install -q kagglehub scikit-learn matplotlib tqdm torchvision

import os
import time
import shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using computing device:", device)


In [ ]:
# Step 3: Download Official FaceForensics++ Extracted Benchmark
import kagglehub

print("Downloading FaceForensics++ Extracted Benchmark (C23 compression)...")
print("This contains 30,000 frames: 5,000 authentic Real + 25,000 across 5 deepfake manipulation techniques.")

dataset_path = kagglehub.dataset_download("fatimahirshad/faceforensics-extracted-dataset-c23")
print("Dataset successfully downloaded to:", dataset_path)
DATASET_PATH = Path(dataset_path)


In [ ]:
# Step 4: Auto-Discover and Balance Manipulation Categories
all_images = [p for p in DATASET_PATH.rglob("*.*") if p.suffix.lower() in [".jpg", ".png", ".jpeg"]]
print(f"Total image frames indexed: {len(all_images)}")

# Category mapping:
# Authentic: Original
# Manipulated: Deepfakes, Face2Face, FaceShifter, FaceSwap, NeuralTextures
categories = {
    "Original (Real)": [],
    "Deepfakes": [],
    "Face2Face": [],
    "FaceShifter": [],
    "FaceSwap": [],
    "NeuralTextures": []
}

for p in all_images:
    posix_str = p.as_posix().lower()
    if "original" in posix_str or "real" in posix_str:
        categories["Original (Real)"].append(p)
    elif "deepfakes" in posix_str:
        categories["Deepfakes"].append(p)
    elif "face2face" in posix_str:
        categories["Face2Face"].append(p)
    elif "faceshifter" in posix_str:
        categories["FaceShifter"].append(p)
    elif "faceswap" in posix_str:
        categories["FaceSwap"].append(p)
    elif "neuraltextures" in posix_str:
        categories["NeuralTextures"].append(p)

for cat_name, file_list in categories.items():
    print(f"  {cat_name:<20}: {len(file_list)} frames")

real_images = categories["Original (Real)"]
# Sample evenly from all 5 manipulation methods to ensure balanced representation
fake_images = []
n_per_fake_cat = min(len(real_images) // 5, 1000)
for cat in ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]:
    cat_files = categories[cat]
    if len(cat_files) > 0:
        chosen = np.random.choice(cat_files, min(len(cat_files), n_per_fake_cat), replace=False)
        fake_images.extend(list(chosen))

print(f"\nBalanced Dataset: {len(real_images[:len(fake_images)])} Real Faces vs {len(fake_images)} Deepfake Faces")

# Visualize sample pairs
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(real_images[0]))
axes[0].set_title(f"Authentic Real Face (YouTube)\n{real_images[0].name}")
axes[0].axis("off")

axes[1].imshow(Image.open(fake_images[0]))
axes[1].set_title(f"FaceForensics++ Deepfake Face\n{fake_images[0].name}")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Step 5: 2D FFT Spectral Extraction Module (GPU-Accelerated)
def compute_fft_2d_torch(image_tensor: torch.Tensor) -> torch.Tensor:
    """
    Computes 2D Fast Fourier Transform log-magnitude spectrum on GPU.
    Natural human camera photography follows P(f) ~ 1/f^2 decay.
    Deepfake generators and neural rendering leave distinctive grid spikes in frequency space.
    """
    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)
    
    # Convert RGB to Grayscale using luminosity weights
    weights = torch.tensor([0.2989, 0.5870, 0.1140], device=image_tensor.device).view(1, 3, 1, 1)
    gray = torch.sum(image_tensor * weights, dim=1, keepdim=True)
    
    # 2D FFT & centered shift
    fft2 = torch.fft.fft2(gray)
    fft_shift = torch.fft.fftshift(fft2, dim=(-2, -1))
    
    # Log magnitude spectrum: log(1 + |F(u, v)|)
    log_mag = torch.log1p(torch.abs(fft_shift))
    
    # Batch min-max normalization to [0, 1]
    B = log_mag.shape[0]
    min_v = log_mag.view(B, -1).min(dim=1)[0].view(B, 1, 1, 1)
    max_v = log_mag.view(B, -1).max(dim=1)[0].view(B, 1, 1, 1)
    denom = torch.clamp(max_v - min_v, min=1e-8)
    
    return (log_mag - min_v) / denom

print("2D FFT Spectral extraction verified.")


In [ ]:
# Step 6: Dual-Stream Network Architecture
class SpectralCNNEncoder(nn.Module):
    """Dedicated frequency encoder for 2D FFT periodic patterns."""
    def __init__(self, out_features=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, out_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
    def forward(self, x):
        return self.fc(self.features(x))


class DualStreamDetector(nn.Module):
    """Dual-Stream Deepfake & AI Image Classifier."""
    def __init__(self, backbone="resnet50", pretrained=True, num_classes=2):
        super().__init__()
        self.backbone_name = backbone
        
        # 1. Spatial Stream (ResNet-50)
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        spatial_dim = resnet.fc.in_features  # 2048
        resnet.fc = nn.Identity()
        self.spatial_encoder = resnet
        
        # 2. Spectral Stream (2D FFT CNN)
        spectral_dim = 256
        self.spectral_encoder = SpectralCNNEncoder(out_features=spectral_dim)
        
        # Auxiliary heads for explainability
        self.spatial_head = nn.Linear(spatial_dim, num_classes)
        self.spectral_head = nn.Linear(spectral_dim, num_classes)
        
        # 3. Fusion Head
        total_dim = spatial_dim + spectral_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x_rgb, return_breakdown=False):
        spatial_emb = self.spatial_encoder(x_rgb)
        spectral_input = compute_fft_2d_torch(x_rgb)
        spectral_emb = self.spectral_encoder(spectral_input)
        
        fused_emb = torch.cat([spatial_emb, spectral_emb], dim=1)
        fused_logits = self.fusion_head(fused_emb)
        
        if return_breakdown:
            return {
                "logits": fused_logits,
                "spatial_logits": self.spatial_head(spatial_emb),
                "spectral_logits": self.spectral_head(spectral_emb)
            }
        return fused_logits

print("DualStreamDetector architecture ready.")


In [ ]:
# Step 7: Create Dataset and Train/Val/Test Splits
class FaceForensicsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Balance dataset to equal size
n_samples = min(len(real_images), len(fake_images), 5000)
np.random.seed(42)

selected_real = list(np.random.choice(real_images, n_samples, replace=False))
selected_fake = list(np.random.choice(fake_images, n_samples, replace=False))

# Label Mapping: 0 = FAKE / DEEPFAKE, 1 = REAL
all_paths = selected_fake + selected_real
all_labels = [0] * len(selected_fake) + [1] * len(selected_real)

indices = np.random.permutation(len(all_paths))
all_paths = [all_paths[i] for i in indices]
all_labels = [all_labels[i] for i in indices]

# 80% Train, 10% Val, 10% Test
n_total = len(all_paths)
n_train = int(0.80 * n_total)
n_val = int(0.10 * n_total)

train_paths, train_labels = all_paths[:n_train], all_labels[:n_train]
val_paths, val_labels = all_paths[n_train:n_train+n_val], all_labels[n_train:n_train+n_val]
test_paths, test_labels = all_paths[n_train+n_val:], all_labels[n_train+n_val:]

print(f"Train samples: {len(train_paths)} (Real: {train_labels.count(1)}, Deepfake: {train_labels.count(0)})")
print(f"Val samples:   {len(val_paths)} (Real: {val_labels.count(1)}, Deepfake: {val_labels.count(0)})")
print(f"Test samples:  {len(test_paths)} (Real: {test_labels.count(1)}, Deepfake: {test_labels.count(0)})")

# Augmentations for video compression and real-world camera noise
img_size = 224
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# drop_last=True prevents single-sample final batch errors
BATCH_SIZE = 64
train_loader = DataLoader(FaceForensicsDataset(train_paths, train_labels, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(FaceForensicsDataset(val_paths, val_labels, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(FaceForensicsDataset(test_paths, test_labels, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"DataLoaders ready ({len(train_loader)} batches per epoch).")


In [ ]:
# Step 8: Multi-Task Training and Evaluation Functions
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    device_type = "cuda" if device.type == "cuda" else "cpu"
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type=device_type, enabled=(device_type == "cuda")):
            out = model(images, return_breakdown=True)
            loss_fusion = criterion(out["logits"], labels)
            loss_spatial = criterion(out["spatial_logits"], labels)
            loss_spectral = criterion(out["spectral_logits"], labels)
            total_loss = loss_fusion + 0.3 * loss_spatial + 0.3 * loss_spectral
            
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += total_loss.item() * images.size(0)
        preds = torch.argmax(out["logits"], dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    device_type = "cuda" if device.type == "cuda" else "cpu"
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast(device_type=device_type, enabled=(device_type == "cuda")):
                out = model(images, return_breakdown=True)
                loss = criterion(out["logits"], labels)
                
            running_loss += loss.item() * images.size(0)
            probs = F.softmax(out["logits"], dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(out["logits"], dim=1).cpu().numpy()
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = running_loss / len(loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
        
    return val_loss, val_acc, precision, recall, f1, auc, all_labels, all_preds, all_probs

print("Training routines compiled.")


In [ ]:
# Step 9: Train Dual-Stream Model on FaceForensics++ (5 Epochs)
SAVE_PATH = "/content/face_deepfake_stream.pth"
ALIAS_PATH = "/content/dual_stream_resnet18_best.pth"

model = DualStreamDetector(backbone="resnet50", pretrained=True, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_val_acc = 0.0
EPOCHS = 5

print(f"Starting FaceForensics++ fine-tuning for {EPOCHS} epochs on {device}...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_acc, prec, rec, f1, auc, _, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    elapsed = time.time() - t0
    
    print(f"Epoch {epoch}/{EPOCHS} [{elapsed:.1f}s] - Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% AUC: {auc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_dict = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_accuracy": float(val_acc),
            "val_auc": float(auc),
            "class_to_idx": {"FAKE": 0, "REAL": 1}
        }
        torch.save(save_dict, SAVE_PATH)
        torch.save(save_dict, ALIAS_PATH)
        print(f"  -> Saved new best checkpoint (Val Acc: {val_acc*100:.2f}%) to {SAVE_PATH}")

print(f"\nTraining complete! Peak FaceForensics++ Validation Accuracy: {best_val_acc*100:.2f}%")


In [ ]:
# Step 10: Independent Test Set Evaluation & Confusion Matrix
checkpoint = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print("Loaded best FaceForensics++ checkpoint from:", SAVE_PATH)

test_loss, test_acc, prec, rec, f1, auc, labels, preds, probs = evaluate(model, test_loader, criterion, device)

print("\n" + "="*55)
print("FINAL FACEFORENSICS++ INDEPENDENT TEST RESULTS:")
print(f"Accuracy:  {test_acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall:    {rec * 100:.2f}%")
print(f"F1 Score:  {f1 * 100:.2f}%")
print(f"ROC-AUC:   {auc:.4f}")
print("="*55)

# Plot Confusion Matrix
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Deepfake", "Real"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap="Blues", ax=ax)
plt.title(f"VeriSight FaceForensics++ Test CM (Acc: {test_acc*100:.1f}%)")
plt.savefig("/content/faceforensics_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Step 11: Download Trained FaceForensics Weights to Your PC
from google.colab import files

print("Downloading face_deepfake_stream.pth to your computer...")
print("Once downloaded, move this file into your project's saved_models/ folder to upgrade your Chrome extension detector!")
files.download(SAVE_PATH)
